
# LifeMate — Qwen3-ASR LoRA **round 2** on 2 × T4 (attention + MLP, 3 epochs)

**Settings (right panel):** Accelerator **GPU T4 ×2** · Internet **On** · add the two private datasets
(`lifemate-data`, `lifemate-code`). `HF_TOKEN` secret optional (Qwen3-ASR is not gated).

**Run:** `MODE = "full"` is already set → **Save Version → Save & Run All (Commit)**, then close the tab. It runs on
Kaggle's servers (~2.5 h; the 12 h limit is far away). Download `out/` from the version's **Output** tab afterwards.

**What it does** (round 1 = yesterday: attention-only LoRA, 200 steps, dev WER 26.97 → 20.53 at A-s100, 19.36 at A-s200):
1. **Model A2:** LoRA r16 on attention **and MLP** of the 28 decoder layers, 3,212 train-partition clips (354 held out),
   **600 steps = 3 epochs**, adapters + holdout loss at 200 / 400 / 600.
2. **Dev transcripts** (887 clips) with each A2 adapter → `out/asr/<cache>/train.jsonl` + WER.
3. **Model B2:** same recipe on **all** 5,036 clips, 941 steps, adapters at 314 / 627 / 941 (same epochs as A2).
   Final-fit candidate only: **never score Model B2 on dev or audit.**
4. **Public transcripts** (1,933 clips) with the B2 adapters that match the **2 best A2 adapters by dev WER**.

If training runs out of GPU memory it is retried once with gradient checkpointing (slower, same result).


In [ ]:
# ---- settings ----------------------------------------------------------------------------------------------
MODE = "full"           # "full" = round 2 (commit directly) | "smoke" = quick interactive end-to-end check
DTYPE = "float32"       # T4 has no bfloat16; float16 can overflow in Qwen layers -> float32 (fast enough here)
STEPS_A, SAVE_A = 600, [200, 400, 600]   # 3 epochs over the 3,212 train-partition clips
MICRO, ACCUM = 2, 4     # per GPU; x 2 GPUs = effective batch 16, the same recipe as the single-GPU pilot
LR, LORA_R, LORA_ALPHA = 5e-5, 16, 32
GRAD_CKPT = False       # switched on automatically if training runs out of memory
RUN_B = True            # Model B: same recipe on ALL train clips (final fit)
PUBLIC = True           # transcribe the public test clips with the best Model B2 checkpoints
PUBLIC_TOP = 2          # how many B2 checkpoints get public transcripts (ranked by the matching A2 dev WER)
TARGETS = "attn+mlp"    # round 2: LoRA on self-attention q/k/v/o AND mlp gate/up/down (round 1: attention only)
NAME_A, NAME_B, TAG = "A2", "B2", "2"   # output folders out/A2, out/B2; transcript tags ftA2-s*, ftB2-s*
ASR_REPO, ASR_REV = "Qwen/Qwen3-ASR-1.7B", "7278e1e70fe206f11671096ffdd38061171dd6e5"

In [ ]:
# ---- locate the two datasets, prepare a writable workspace ---------------------------------------------------
import os, sys, json, shutil, subprocess, time
from pathlib import Path

INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
OUT = Path(os.environ.get("LIFEMATE_KAGGLE_OUT", "/kaggle/working/out"))
WORK = Path(os.environ.get("LIFEMATE_KAGGLE_WORK", "/tmp/lifemate"))

def _find_dirs(pred):
    hits = []
    for dp, dns, fns in os.walk(INPUT, followlinks=True):
        dns[:] = [d for d in dns if d not in ("__MACOSX", "audio", "train_environment", "public_test_environment")]
        if pred(Path(dp), fns):
            hits.append(Path(dp))
    return sorted(hits, key=lambda p: len(p.parts))

data_hits = _find_dirs(lambda p, fns: (p / "train" / "train.jsonl").exists() and (p / "train" / "audio").is_dir())
code_hits = _find_dirs(lambda p, fns: (p / "lifemate" / "__init__.py").exists() and (p / "kaggle" / "asr_ddp.py").exists())
assert data_hits, f"no folder with train/train.jsonl + train/audio/ under {INPUT}"
assert code_hits, f"no folder with lifemate/ + kaggle/asr_ddp.py under {INPUT} (attach the code dataset)"
DATA, CODE_SRC = data_hits[0], code_hits[0]

CODE = WORK / "zalo"
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "vm", "dist"))
ATTN_MLP = r"thinker\.model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
if TARGETS == "attn+mlp":   # patch the working copy only; the training script reads TARGET_MODULES from here
    _tl = CODE / "scripts" / "train_asr_lora.py"
    _src = _tl.read_text()
    _old = 'TARGET_MODULES = r"thinker\\.model\\.layers\\.\\d+\\.self_attn\\.(q_proj|k_proj|v_proj|o_proj)"'
    assert _old in _src, "TARGET_MODULES line not found in train_asr_lora.py"
    _tl.write_text(_src.replace(_old, "TARGET_MODULES = r\"" + ATTN_MLP + "\""))
    print("LoRA targets patched to attention + MLP:", ATTN_MLP)
ART = CODE / "artifacts"
for d in (OUT, OUT / "logs", WORK / "shards"):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(LIFEMATE_DATA_ROOT=str(DATA), LIFEMATE_ARTIFACTS=str(ART), HF_HOME=str(WORK / "hf"),
                  TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1")
sys.path.insert(0, str(CODE))

n_train_wav = sum(1 for _ in (DATA / "train" / "audio").rglob("*.wav"))
has_public = (DATA / "public_test" / "public_test.jsonl").exists() and (DATA / "public_test" / "audio").is_dir()
print("data :", DATA, f"| train wav {n_train_wav} | public_test {'yes' if has_public else 'NO'}")
print("code :", CODE_SRC, "->", CODE)
assert n_train_wav == 5036, f"expected 5,036 train clips, found {n_train_wav} (incomplete upload?)"
if PUBLIC and not has_public:
    print("public_test not attached -> PUBLIC = False"); PUBLIC = False

In [ ]:
# ---- helpers: run a command with live output + a log file; run one command per GPU in parallel ------------------
LOGS = OUT / "logs"
_QUIET = ("Setting `pad_token_id`", "The following generation flags")

def run(cmd, log, env=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    with open(LOGS / log, "a") as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=CODE,
                             env=env or os.environ.copy())
        for line in p.stdout:
            lf.write(line); lf.flush()
            if not line.startswith(_QUIET):
                print(line, end="", flush=True)
        rc = p.wait()
    if rc:
        raise RuntimeError(f"{log}: exit code {rc} — see {LOGS / log}")

def run_parallel(jobs, poll=30):
    # jobs: [(cmd, log name, gpu index)]
    procs = []
    for cmd, log, gpu in jobs:
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        lf = open(LOGS / log, "a")
        print(f"[gpu {gpu}] $", " ".join(map(str, cmd)), flush=True)
        procs.append((subprocess.Popen([str(c) for c in cmd], stdout=lf, stderr=subprocess.STDOUT, text=True,
                                       cwd=CODE, env=env), log, lf))
    def tail(log):
        lines = [l for l in open(LOGS / log, errors="replace").read().splitlines() if l.strip() and not l.startswith(_QUIET)]
        return lines[-1][:160] if lines else ""
    while any(p.poll() is None for p, _, _ in procs):
        time.sleep(poll)
        print(" | ".join(f"{log}: {tail(log)}" for _, log, _ in procs), flush=True)
    for p, log, lf in procs:
        lf.close()
        if p.returncode:
            print(open(LOGS / log, errors="replace").read()[-3000:])
            raise RuntimeError(f"{log}: exit code {p.returncode}")

def section(title):
    print(f"\n===== {time.strftime('%H:%M:%S')} {title} =====", flush=True)

T0 = time.time()

In [ ]:
# ---- install qwen-asr without touching Kaggle's CUDA torch ---------------------------------------------------------
import importlib.metadata as md
section("install")
pins = []
for pkg in ("torch", "torchvision", "torchaudio"):
    try:
        pins.append(f"{pkg}=={md.version(pkg).split('+')[0]}")
    except md.PackageNotFoundError:
        pass
Path("/tmp/constraints.txt").write_text("\n".join(pins) + "\n")
print("keeping:", pins)
pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
r = subprocess.run(pip + ["qwen-asr==0.0.6", "peft>=0.17,<0.19", "-c", "/tmp/constraints.txt"])
if r.returncode:
    print("full install failed -> minimal install (no gradio/flask/demo extras)")
    subprocess.run(pip + ["--no-deps", "qwen-asr==0.0.6"], check=True)
    subprocess.run(pip + ["transformers==4.57.6", "accelerate==1.12.0", "peft>=0.17,<0.19", "qwen-omni-utils",
                          "librosa", "soundfile", "pytz", "sox", "-c", "/tmp/constraints.txt"], check=True)

CHECK = r'''
import torch, transformers, peft
from qwen_asr import Qwen3ASRModel
n = torch.cuda.device_count()
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("gpus:", n, [torch.cuda.get_device_name(i) for i in range(n)])
'''
r = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
if r.returncode and ("nagisa" in r.stderr or "soynlp" in r.stderr):
    # only the forced aligner (Japanese/Korean word splitting) needs these; install, else stub them out
    if subprocess.run(pip + ["nagisa==0.2.11", "soynlp==0.0.493"]).returncode:
        stub = WORK / "stubs"
        for m in ("nagisa", "soynlp"):
            (stub / m).mkdir(parents=True, exist_ok=True); (stub / m / "__init__.py").write_text("")
        os.environ["PYTHONPATH"] = f"{stub}:{os.environ.get('PYTHONPATH', '')}"
        print("stubbed nagisa/soynlp (unused by ASR)")
    r = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True, env=os.environ.copy())
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0, "qwen-asr import failed (see above)"
NGPU = int(r.stdout.split("gpus:")[1].split()[0])
torch_now = r.stdout.split()[1].split("+")[0]
assert pins and pins[0] == f"torch=={torch_now}", f"pip replaced Kaggle's torch: {pins[0]} -> {torch_now}"
if NGPU < 2:
    print(f"WARNING: {NGPU} GPU visible — set Accelerator to 'GPU T4 x2'. Continuing on {NGPU}.")
NGPU = max(1, NGPU)
subprocess.run(f"{sys.executable} -m pip freeze > {OUT / 'pip_freeze.txt'}", shell=True)

In [ ]:
# ---- download Qwen3-ASR once (both training processes then load it from disk) ---------------------------------------
section("download Qwen3-ASR")
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")      # optional: Add-ons -> Secrets -> HF_TOKEN
except Exception:
    pass
if token:
    os.environ["HF_TOKEN"] = token
r = subprocess.run([sys.executable, "-c", f"from huggingface_hub import snapshot_download; "
                    f"print(snapshot_download({ASR_REPO!r}, revision={ASR_REV!r}))"],
                   capture_output=True, text=True, check=True, env=os.environ.copy())
MODEL = Path(r.stdout.strip().splitlines()[-1])
print("model:", MODEL, f"{sum(f.stat().st_size for f in MODEL.rglob('*') if f.is_file()) / 2**30:.1f} GB")

In [ ]:
# ---- manifests: A = train partition (from the code dataset, checked), B = all train clips --------------------------
section("manifests")
from lifemate.config import load_config
from lifemate.asr.base import backend_name
from lifemate.data.loader import load_samples, gold_transcripts
from lifemate.data.splits import load_splits

split = load_splits(ART / "splits" / "v1.json")
MAN_A, MAN_B = ART / "asr_ft", ART / "asr_ft_all"
rows_a = [json.loads(l) for l in open(MAN_A / "train.jsonl", encoding="utf-8")]
rows_h = [json.loads(l) for l in open(MAN_A / "holdout.jsonl", encoding="utf-8")]
held_out = set(split["dev"]) | set(split["audit"])
assert not {r["sample_id"] for r in rows_a + rows_h} & held_out, "dev/audit clips in Model A's manifest"
run([sys.executable, "kaggle/build_all_manifest.py", "--out", MAN_B], "manifest_b.log")
n_a, n_b = len(rows_a), sum(1 for _ in open(MAN_B / "train.jsonl", encoding="utf-8"))
STEPS_B = round(STEPS_A * n_b / n_a)
SAVE_B = [round(s * n_b / n_a) for s in SAVE_A]
print(f"Model A: {n_a} train + {len(rows_h)} holdout clips, {STEPS_A} steps, saves {SAVE_A}")
print(f"Model B: {n_b} clips (all), {STEPS_B} steps, saves {SAVE_B}  (same single pass over its data)")

GOLD = {k: v for s in load_samples(DATA, "train") for k, v in gold_transcripts(s).items()}
CACHE_CFG = {**{f"ftA{TAG}-s{s}": None for s in SAVE_A}, **{f"ftB{TAG}-s{s}": None for s in SAVE_B}}
_base = backend_name(load_config(CODE / "configs" / "today_names.yaml"))      # ...-7278e1e-ctxnames
CACHE_NAME = {k: _base.replace("-ctxnames", f"-{k}-ctxnames") for k in CACHE_CFG}   # = backend_name with asr.tag k
print(json.dumps(CACHE_NAME, indent=1))

In [ ]:
# ---- shared steps: train on all GPUs, transcribe with one shard per GPU, WER ----------------------------------------
import re, unicodedata

def train(name, manifest, steps, save, extra=(), limit=0):
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NGPU}", "kaggle/asr_ddp.py",
           "--model", MODEL, "--manifest", manifest, "--out", OUT / name, "--steps", steps, "--save", *save,
           "--micro", MICRO, "--accum", ACCUM, "--lr", LR, "--r", LORA_R, "--alpha", LORA_ALPHA, "--dtype", DTYPE,
           "--context_mode", "names", *extra]
    if GRAD_CKPT:
        cmd.append("--grad_ckpt")
    if limit:
        cmd += ["--limit", limit]
    run(cmd, f"train_{name}.log")

def transcribe(tag, adapter, split_name, part, dest, limit=0):
    shards = [WORK / "shards" / f"{tag}.{split_name}.{part}.{g}.jsonl" for g in range(NGPU)]
    jobs = []
    for g, sh in enumerate(shards):
        cmd = [sys.executable, "kaggle/asr_transcribe.py", "--model", MODEL, "--dtype", DTYPE, "--split", split_name,
               "--part", part, "--context_mode", "names", "--shard", g, "--nshards", NGPU, "--out", sh]
        if adapter:
            cmd += ["--adapter", adapter]
        if limit:
            cmd += ["--limit", limit]
        jobs.append((cmd, f"asr_{tag}_{split_name}_{part}_gpu{g}.log", g))
    run_parallel(jobs)
    dest.parent.mkdir(parents=True, exist_ok=True)
    rows = [l for sh in shards for l in open(sh, encoding="utf-8") if l.strip()]
    dest.write_text("".join(rows), encoding="utf-8")
    print(f"{tag}: {len(rows)} transcripts -> {dest}")
    return {json.loads(l)["audio"]: json.loads(l)["text"] for l in rows}

def _words(t):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", t).lower())

def _ed(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def wer(hyp):
    e = n = 0
    for p, h in hyp.items():
        g = _words(GOLD[p]); e += _ed(g, _words(h)); n += len(g)
    return round(100 * e / max(n, 1), 2)

def train_safe(name, manifest, steps, save, extra=()):
    """Train; if the GPU runs out of memory, retry once from scratch with gradient checkpointing."""
    global GRAD_CKPT
    try:
        train(name, manifest, steps, save, extra)
    except RuntimeError:
        log = (LOGS / f"train_{name}.log").read_text(errors="replace")
        if GRAD_CKPT or "out of memory" not in log.lower():
            raise
        print(f"{name}: out of GPU memory -> retrying with gradient checkpointing", flush=True)
        RESULTS["notes"].append(f"{name}: OOM without gradient checkpointing, retried with it")
        GRAD_CKPT = True
        shutil.rmtree(OUT / name, ignore_errors=True)
        (LOGS / f"train_{name}.log").rename(LOGS / f"train_{name}.oom.log")
        train(name, manifest, steps, save, extra)

RESULTS = {"wer_dev": {}, "notes": [], "round": 2, "targets": TARGETS, "steps_a": STEPS_A, "save_a": SAVE_A}

In [ ]:
# ---- SMOKE (MODE = "smoke"): tiny run of every step on both GPUs, then stop -----------------------------------------
if MODE == "smoke":
    section("smoke: training 4 steps on 32 clips")
    train("smoke_A", MAN_A, 4, [2, 4], limit=32)
    section("smoke: transcribing 8 dev clips per GPU, base and adapter")
    hb = transcribe("smoke_base", None, "train", "dev", WORK / "smoke" / "base.jsonl", limit=8)
    ha = transcribe("smoke_A4", OUT / "smoke_A" / "step4", "train", "dev", WORK / "smoke" / "a4.jsonl", limit=8)
    for p in list(hb)[:3]:
        print(f"gold : {GOLD[p]}\nbase : {hb[p]}\nA s4 : {ha[p]}\n")
    print(f"SMOKE OK in {(time.time() - T0) / 60:.1f} min — check the memory (mem ...G) lines above, then set "
          f"MODE = 'full' and Save Version -> Save & Run All.")
    shutil.rmtree(OUT / "smoke_A", ignore_errors=True)

In [ ]:
# ---- 1. Model A: train partition only ----------------------------------------------------------------------------
if MODE == "full":
    section(f"Model {NAME_A} training (train partition, {TARGETS})")
    train_safe(NAME_A, MAN_A, STEPS_A, SAVE_A)

In [ ]:
# ---- 2. dev transcripts + WER: base vs A-step100 vs A-step200 --------------------------------------------------------
if MODE == "full":
    for tag, adapter in [(f"ftA{TAG}-s{s}", OUT / NAME_A / f"step{s}") for s in SAVE_A]:   # base / round 1 known already
        section(f"dev transcripts: {tag}")
        hyp = transcribe(tag, adapter, "train", "dev", OUT / "asr" / CACHE_NAME[tag] / "train.jsonl")
        RESULTS["wer_dev"][tag] = wer(hyp)
        print(f"dev WER {tag}: {RESULTS['wer_dev'][tag]}%")
    best = min((t for t in RESULTS["wer_dev"] if t.startswith("ftA")), key=RESULTS["wer_dev"].get)
    RESULTS["best_A_by_dev_wer"] = best
    print("dev WER:", RESULTS["wer_dev"], "| best A2 by WER:", best, "| round 1: base 26.97, ftA-s100 20.53, ftA-s200 19.36")
    (OUT / "results.json").write_text(json.dumps(RESULTS, indent=1))

In [ ]:
# ---- 3. Model B: all train clips (final-fit candidate; never evaluate it on dev/audit) ------------------------------
if MODE == "full" and RUN_B:
    section(f"Model {NAME_B} training (ALL train clips, {TARGETS})")
    train_safe(NAME_B, MAN_B, STEPS_B, SAVE_B, extra=["--no_holdout"])

In [ ]:
# ---- 4. public test transcripts with both Model B checkpoints --------------------------------------------------------
if MODE == "full" and RUN_B and PUBLIC:
    rank = sorted(range(len(SAVE_A)), key=lambda k: RESULTS["wer_dev"][f"ftA{TAG}-s{SAVE_A[k]}"])   # best A2 first
    RESULTS["public_b"] = []
    for k in rank[:PUBLIC_TOP]:
        s = SAVE_B[k]
        tag = f"ftB{TAG}-s{s}"
        section(f"public transcripts: {tag} (matches ftA{TAG}-s{SAVE_A[k]}, dev WER {RESULTS['wer_dev'][f'ftA{TAG}-s{SAVE_A[k]}']})")
        transcribe(tag, OUT / NAME_B / f"step{s}", "public_test", "all", OUT / "asr" / CACHE_NAME[tag] / "public_test.jsonl")
        RESULTS["public_b"].append({"tag": tag, "matches": f"ftA{TAG}-s{SAVE_A[k]}"})
        (OUT / "results.json").write_text(json.dumps(RESULTS, indent=1))

In [ ]:
# ---- summary -------------------------------------------------------------------------------------------------------
section("summary")
RESULTS["minutes_total"] = round((time.time() - T0) / 60, 1)
for name in (NAME_A, NAME_B):
    h = OUT / name / "history.json"
    if h.exists():
        RESULTS[f"history_{name}"] = {k: v for k, v in json.loads(h.read_text()).items() if k != "loss"}
(OUT / "results.json").write_text(json.dumps(RESULTS, indent=1))
lines = [f"# Kaggle ASR round 2 ({MODE}, LoRA {TARGETS}, {STEPS_A} steps) — {time.strftime('%Y-%m-%d %H:%M')}", "",
         f"- total {RESULTS['minutes_total']} min on {NGPU} GPU(s), dtype {DTYPE}, effective batch {MICRO * ACCUM * NGPU}",
         f"- dev WER (names context): {RESULTS.get('wer_dev')}", f"- best A2 by dev WER: {RESULTS.get('best_A_by_dev_wer')} (round 1: base 26.97, ftA-s100 20.53, ftA-s200 19.36)",
         f"- public transcripts: {RESULTS.get('public_b')}", f"- notes: {RESULTS.get('notes')}",
         "- Model B = all train clips incl. dev/audit: final fit only, never score it on dev/audit", "",
         "Next: put out/asr/* on the Modal volume under /artifacts/asr/ and score the v2 agent on dev C with the best ftA2 "
         "transcripts (configs/v2_ftA2_s<step>.yaml); adopt round 2 only if dev C beats the ftA-s100 transcripts.", "", "## files"]
size = 0
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        size += f.stat().st_size
        lines.append(f"- {f.relative_to(OUT)} ({f.stat().st_size / 2**20:.1f} MB)")
(OUT / "SUMMARY.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines[:8]))
print(f"output size {size / 2**30:.2f} GB (Kaggle keeps up to 20 GB)")
assert size < 19 * 2**30